# Post-processing: frame potential + training results

One notebook covering both halves of the project so they can be compared
directly: how close each `circuit_set` architecture is to a unitary
`t`-design (`frame_potential.csv`), and how well it actually trains on
Fourier-series targets (`experiments.csv` + `costs/*.npy`). The last two
sections join the two and ask whether the frame-potential gap is a better
(or additional, beyond raw parameter count) indicator of trainability.

`reps` (frame potential) and `anzats_reps` (training) are the same
parameter, passed straight through to the same `circuit_set(num, reps)` --
`frame_potential.py` and `tracking.py` both call
`circuits.n_trainable(num, n_qubits, reps)` with identical semantics (no
off-by-one between them), so the two are joined directly without an
offset.

In [ ]:
import sys
# repo root, so frame_potential/circuits/tracking import the same way the CLI does
sys.path.append("..")

In [ ]:
import os

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import pandas as pd
import mplcursors
from scipy.stats import spearmanr, pearsonr, rankdata, linregress

import frame_potential as fp
import circuits
import generic_ansatz
import tracking

# uncomment to be able to interact with plots in jupyter notebooks
%matplotlib widget

## Part A -- Frame potential

### Load

`n_params` isn't a column in the CSV -- it's derived per row from
`(circuit_num, n_qubits, reps)` via `circuits.n_trainable`, computed once
per unique triple and cached. A row's `circuit_num` can only be resolved if
that number is still defined in `circuits.py` today; unresolved rows stay in
`df_fp` but are excluded from the parameter-count plots.

In [ ]:
path = "../results/remote/frame_potential.csv"
df_fp = fp.load_frame_potential(path=path)
print(f"{len(df_fp)} rows, {df_fp.groupby(['circuit_num', 'n_qubits', 'reps', 't']).ngroups} "
      "unique (circuit_num, n_qubits, reps, t) configs")
df_fp.head()

In [ ]:
# delete the rows that have not converged (i.e. fidelity_error is bigger than delta)
df_fp = df_fp[df_fp["fidelity_error"] <= df_fp["delta"]*2]
df_fp = df_fp[df_fp["delta"] < 1000]

n_qubits = 6
#select only the rows with n_qubits= n_qubits
df_fp = df_fp[df_fp['n_qubits'] == n_qubits]
print(f"{len(df_fp)} rows, {df_fp.groupby(['circuit_num', 'n_qubits', 'reps', 't']).ngroups} "
      "unique (circuit_num, n_qubits, reps, t) configs")

### Raw scatter: every run, faceted by `t`

Color is `circuit_num`; hover a point for the exact `(circuit_num, n_qubits,
reps)`.

In [ ]:
fp_plottable = df_fp.dropna(subset=["n_params"]).copy()

circuit_nums_fp = sorted(fp_plottable["circuit_num"].unique())
palette_fp = {c: cm.tab10(i % 10) for i, c in enumerate(circuit_nums_fp)}
t_values = sorted(fp_plottable["t"].unique())

plt.close("all")
print(f"plotting {len(circuit_nums_fp)} circuits, {len(t_values)} t values, "
      f"{len(fp_plottable)} rows of data (out of {len(df_fp)} total rows)")
fig, axes = plt.subplots(1, len(t_values), figsize=(9 * len(t_values), 6), squeeze=False)
axes = axes[0]

handles_a = []
for ax, t in zip(axes, t_values):
    sub = fp_plottable[fp_plottable["t"] == t]
    for c in circuit_nums_fp:
        pts = sub[sub["circuit_num"] == c]
        if len(pts):
            ax.plot(pts["n_params"], pts["delta"], color=palette_fp[c], alpha=0.8)
            h = ax.scatter(pts["n_params"], pts["delta"], color=palette_fp[c], alpha=0.8)
            handles_a.append((h, pts))
    ax.set_yscale("log")
    ax.set_xlabel("number of parameters",fontsize=20)
    ax.set_ylabel(f"$\\Delta F^{{({t})}}$",fontsize=20)
    ax.set_title(f"t={t}, n_qubits={n_qubits}",fontsize=20)


fig.suptitle("Delta frame potential vs. number of parameters",fontsize=25)
fig.tight_layout()
cursor_a = mplcursors.cursor([h for h, _ in handles_a], hover=True)

@cursor_a.connect("add")
def _(sel):
    pts = next(p for h, p in handles_a if h is sel.artist)
    row = pts.iloc[sel.index]
    sel.annotation.set_text(
        f"circuit {row.circuit_num}  n_qubits={row.n_qubits}  reps={row.reps}  t={row.t}\n"
        f"n_params={row.n_params}  delta={row.delta:.3g}"
    )

plt.show()

### k = per-circuit scaling regression, not a single division

For each `circuit_num`, across all its available `reps` values, fit a
linear regression of `log(delta)` against `n_params`:

    log(delta) ~= intercept - k * n_params

`delta` (not `frame_potential` directly) is the right target for this: it
goes to 0 as an architecture becomes a genuine 2-design, so an idealized
exponential approach to Haar (`delta ~ A * exp(-k * n_params)`) is exactly
a straight line in log-delta space -- `frame_potential` itself plateaus
toward the (nonzero) Haar value instead of 0, so it doesn't have that same
clean log-linear form. `k` is defined as the *negative* of the fitted
slope, so a bigger `k` means faster convergence per parameter -- same sign
convention as the earlier naive version.

`r_squared` is how well that straight-line assumption actually held for
that circuit; `n_points` is how many `(n_params, delta)` pairs went into
the fit. A circuit needs at least 2 *distinct* `n_params` values to
regress at all (excluded otherwise), and exactly 2 points gives a trivial,
meaningless `r_squared = 1` (a line always fits 2 points perfectly) --
`n_points` is in the hover text specifically so those don't get mistaken
for a real result.

Two panels, side by side, grouping the same `k` values two different ways:
by `architecture` (the `(topo_category, topo_structure)` pair
`get_architecture_from_code` extracts -- rotation arity and gate choice
collapse together within a group, isolating whether *topology* changes how
k scales) and by `rotation_arity` (the code's first digit -- 1/RY, 2/RX+RZ,
3/full Rot -- collapsing topology and gate choice instead, isolating
whether the *rotation layer* does). Each group gets its own x-slot (jitter
within it only for point visibility, not meaning) and its own color;
`display_median` (off by default) overlays a black-cored median bar per
group for an even quicker at-a-glance comparison. Hover still gives the
exact circuit/k/R²/n_points/p for any point, in either panel.</cell id="5bbdfc2c">


In [ ]:
def get_architecture_from_code(code):
    a = code // 100
    a = a % 100
    return a


def get_rotation_arity_from_code(code):
    """First digit of the 5-digit generic_ansatz code: how many angles per
    wire per rotation layer (1=RY, 2=RX+RZ, 3=full Rot) -- see
    generic_ansatz.py's module docstring."""
    return code // 10000

In [ ]:
rows = []
for circuit_num, group in fp_plottable.groupby("circuit_num"):
    group = group.dropna(subset=["n_params", "delta"])
    group = group[group["delta"] > 0]  # log(delta) undefined/meaningless at or below 0
    x = group["n_params"].to_numpy(dtype=float)
    y = np.log(group["delta"].to_numpy(dtype=float))
    if len(np.unique(x)) < 2:
        continue  # a straight line needs at least 2 distinct n_params values
    res = linregress(x, y)
    rows.append({"circuit_num": circuit_num, "k": -res.slope, "r_squared": res.rvalue ** 2,
                 "intercept": res.intercept, "p_value": res.pvalue, "n_points": len(x),
                 "architecture": get_architecture_from_code(circuit_num),
                 "rotation_arity": get_rotation_arity_from_code(circuit_num),
                 # the n_params range the fit was actually built on -- needed later to tell
                 # a real estimate (inside this range) from an extrapolation (outside it)
                 "x_min": x.min(), "x_max": x.max()})

k_df = pd.DataFrame(rows)
print(f"{len(k_df)}/{fp_plottable['circuit_num'].nunique()} circuits regressed "
      f"(the rest had fewer than 2 distinct n_params values)")
print(f"{(k_df['n_points'] == 2).sum()} of those are 2-point fits -- r_squared=1 there "
      "is trivial, not evidence")
k_df.sort_values("k", ascending=False).head(10)

In [ ]:
def architecture_label(a):
    """Readable name for the 2-digit (topo_category, topo_structure) group
    get_architecture_from_code extracts -- see generic_ansatz.py's module
    docstring for what each digit means."""
    names = {
        0: "no topology",
        10: "chain, sequential",
        11: "chain, brickwork",
        12: "chain, double brickwork",
        13: "circular, alternating",
        14: "circular",
        20: "all-to-all",
    }
    return names.get(a, f"architecture {a}")


def rotation_arity_label(r):
    """Readable name for the rotation_arity group (1st digit of the code)."""
    names = {1: "1 (RY)", 2: "2 (RX+RZ)", 3: "3 (full Rot)"}
    return names.get(r, str(r))


display_median = False


def plot_k_by_group(ax, group_col, label_fn, title, rng):
    """One panel of the k-vs-group comparison: jittered points within each
    group's own x-slot, colored by group. Shared by both panels below (the
    topology grouping and the rotation-arity one) so they're drawn
    identically and only differ in which column groups them."""
    groups = sorted(k_df[group_col].unique())
    x_of = {g: i for i, g in enumerate(groups)}
    palette = {g: cm.tab10(i % 10) for i, g in enumerate(groups)}

    jitter = rng.uniform(-0.3, 0.3, size=len(k_df))
    x_pos = k_df[group_col].map(x_of) + jitter
    colors = [palette[g] for g in k_df[group_col]]
    sc = ax.scatter(x_pos, k_df["k"], c=colors, alpha=0.85, edgecolor="black",
                     linewidth=0.3, s=50, zorder=2)

    if display_median:
        # one median bar per group -- the at-a-glance cross-group comparison
        # a jittered point cloud alone doesn't give you
        for g, x in x_of.items():
            med = k_df.loc[k_df[group_col] == g, "k"].median()
            ax.plot([x - 0.35, x + 0.35], [med, med], color=palette[g], linewidth=4, zorder=3)
            ax.plot([x - 0.35, x + 0.35], [med, med], color="black", linewidth=1, zorder=4)

    ax.set_xticks(list(x_of.values()))
    ax.set_xticklabels(
        [f"{label_fn(g)}\n(n={(k_df[group_col] == g).sum()})" for g in groups],
        rotation=20, ha="right", fontsize=10,
    )
    ax.axhline(0, color="black", linewidth=0.5, zorder=0)
    ax.set_xlim(-0.6, len(groups) - 0.4)
    ax.set_title(title, fontsize=14)
    return sc


rng = np.random.default_rng(0)
plt.close("all")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

sc1 = plot_k_by_group(ax1, "architecture", architecture_label, "By topology", rng)
ax1.set_ylabel("$\\kappa$ = $-$slope of $\\log(\\Delta F^{(k)})$ vs. $N_p$", fontsize=13)

sc2 = plot_k_by_group(ax2, "rotation_arity", rotation_arity_label, "By rotation arity", rng)

fig.suptitle("Per-circuit scaling coefficient $\\kappa$", fontsize=16)
fig.tight_layout()

panels = [(sc1, "architecture", architecture_label), (sc2, "rotation_arity", rotation_arity_label)]
cursor_k = mplcursors.cursor([sc for sc, _, _ in panels], hover=True)

@cursor_k.connect("add")
def _(sel):
    _, group_col, label_fn = next(p for p in panels if p[0] is sel.artist)
    row = k_df.iloc[sel.index]
    sel.annotation.set_text(
        f"circuit {row.circuit_num}  ({label_fn(row[group_col])})\n"
        f"$\\kappa$={row.k:.4g}  R²={row.r_squared:.3f}  n_points={row.n_points}  p={row.p_value:.2g}"
    )

plt.show()

### Best circuits by the k-criteria

A circuit counts as "promising" here if its Δ*F*^(t) convergence is both
**fast** and **trustworthy**:

- `k > 0.10` -- Δ*F*^(t) shrinks quickly per added parameter (see the
  regression above).
- `r_squared > 0.9` -- the log-linear fit that `k` comes from actually
  held up well, not an artifact of a handful of noisy points.
- `n_points > 2` -- more than the bare minimum 2-point line, which fits
  perfectly (`r_squared = 1`) by construction and would otherwise pass the
  `r_squared` filter for free without being real evidence.

Thresholds are named constants below (not inlined in the filter) so they're
easy to retune from one place. `The_good_ones` (the filtered DataFrame,
keyed by `circuit_num`) is reused later, in Part C's cost-vs-frame-potential
plot, to highlight exactly these architectures there too.

In [ ]:
MIN_K = 0.10
MIN_R2 = 0.9
MIN_N_POINTS = 2  # strictly greater than -- excludes trivial 2-point (r_squared=1 by construction) fits

The_good_ones = k_df[
    (k_df["k"] > MIN_K) & (k_df["r_squared"] > MIN_R2) & (k_df["n_points"] > MIN_N_POINTS)
].copy()
The_good_ones["topology"] = The_good_ones["architecture"].map(architecture_label)
The_good_ones["rotation"] = The_good_ones["rotation_arity"].map(rotation_arity_label)

display_cols = ["circuit_num", "topology", "rotation", "k", "r_squared", "n_points", "p_value"]
summary = The_good_ones[display_cols].sort_values("k", ascending=False).reset_index(drop=True)

print(f"{len(summary)}/{len(k_df)} circuits pass k>{MIN_K}, R²>{MIN_R2}, n_points>{MIN_N_POINTS}")
summary.style.format({"k": "{:.4f}", "r_squared": "{:.3f}", "p_value": "{:.2g}"}) \
    .background_gradient(subset=["k"], cmap="Greens") \
    .set_caption("Most promising architectures (fast + trustworthy ΔF^(t) convergence)")

## Part B -- Training results

Loads `results/experiments.csv` + `results/costs/*.npy` (local), and the
same thing from `results/remote/` if `pull_remote_results.sh` has pulled a
remote run down -- see that script (gitignored, personal, not tracked).
Local and remote are tagged with a `source` column rather than merged into
one CSV, since both are append-only logs that would otherwise clobber each
other's rows on a pull.

In [ ]:
def load_results(path, source_label):
    """Return (df_exp, df_costs) for one results/ directory, tagged with source_label."""
    df_exp = pd.DataFrame(tracking.load_experiments(path=path))
    df_exp["source"] = source_label

    df_costs = pd.DataFrame(tracking.load_costs(path=path))
    df_costs["source"] = source_label
    return df_exp, df_costs


SOURCES = [("../results/remote", "remote")]

exp_frames, cost_frames = [], []
for path, label in SOURCES:
    if not os.path.isdir(path):
        print(f"skipping '{label}': {path} does not exist (nothing there yet)")
        continue
    df_exp_i, df_costs_i = load_results(path, label)
    if len(df_exp_i):
        exp_frames.append(df_exp_i)
        cost_frames.append(df_costs_i)
    print(f"{label}: {len(df_exp_i)} experiments, {len(df_costs_i)} cost curves ({path})")

df_exp = pd.concat(exp_frames, ignore_index=True)
df_costs = pd.concat(cost_frames, ignore_index=True)
df_exp.head()

### Types

`pd.to_numeric(..., errors="coerce")` rather than a plain `.astype` --
local and remote can come from different commits of this repo (e.g. a
remote run from before some column was added), so a merged column isn't
guaranteed to be uniformly numeric-looking; a stray empty string becomes
`NaN` instead of crashing the whole load.

In [ ]:
int_cols = ["circuit_num", "n_qubits", "layers", "anzats_reps", "n_params", "max_steps", "batch_size"]
float_cols = ["final_cost", "initial_cost", "n_expvals"]

for col in int_cols:
    df_exp[col] = pd.to_numeric(df_exp[col], errors="coerce").astype("Int64")
for col in float_cols:
    df_exp[col] = pd.to_numeric(df_exp[col], errors="coerce")

df_exp["param_per_layer"] = df_exp["n_params"] / df_exp["layers"]

n_bad = df_exp["n_params"].isna().sum()
if n_bad:
    print(f"{n_bad} row(s) have an unparseable n_params -- excluded from the plots below.")


# keep only the rows with n_qubits= n_qubits
df_exp = df_exp[df_exp['n_qubits'] == n_qubits]

df_exp.head()

### Final cost: averaged over the last 10 steps, not just the last batch

`final_cost` in `experiments.csv` is the *single* last training step's own
batch loss (see `tracking.record_training` -- `cst[-1]`, never a
separate full-dataset evaluation). With `batch_size=100` out of
`n_train_samples=800`, that's one noisy batch standing in for "how well
did this converge", not the model's actual loss. Averaging the last
`LAST_N_STEPS` entries of the full cost curve (`df_costs`) cancels most of
that batch noise out, without needing the full dataset's loss at every
step. `final_cost_avg10` feeds the architecture-level aggregation below in
place of raw `final_cost` -- the median relative difference between the
two is ~18% in this dataset, so this isn't a rounding correction.

In [ ]:
LAST_N_STEPS = 10  # average the tail of the cost curve instead of trusting one (noisy) last-batch loss

cost_lookup = df_costs.set_index("experiment_id")["costs"].apply(lambda c: np.mean(c[-LAST_N_STEPS:]))
df_exp["final_cost_avg10"] = df_exp["id"].map(cost_lookup)

n_missing = df_exp["final_cost_avg10"].isna().sum()
if n_missing:
    print(f"{n_missing} experiment(s) have no matching cost curve in df_costs -- final_cost_avg10 is NaN there")

rel_diff = (df_exp["final_cost_avg10"] - df_exp["final_cost"]).abs() / df_exp["final_cost"]
print(f"final_cost_avg10 vs. raw final_cost: median relative difference {rel_diff.median():.1%} "
      f"(max {rel_diff.max():.1%}) -- confirms the single last batch is not a reliable estimate on its own")

df_exp[["id", "circuit_num", "anzats_reps", "final_cost", "final_cost_avg10"]].head()

### Mean over problem instances

Each row is one training run of one architecture on one randomly-drawn
target Fourier function -- a "problem instance". A single run's
`final_cost_avg10` (the last-10-step average above, not the raw
single-batch `final_cost`) says more about that particular target than
about the architecture, so for an architecture-level (non-specific)
conclusion we pool it over every run sharing the same `(circuit_num,
n_qubits, layers, anzats_reps)` -- the full model spec, held fixed;
`source` is *not* part of this key, since local and remote runs of the
same config are just more draws of the same problem-instance distribution,
not a different quantity.

`n_runs` says how many instances actually back each point -- most of the
current dataset still has `n_runs == 1` (a sweep that hasn't been repeated
yet), in which case `sem_final_cost` is 0 (no error bar, not because the
true error is zero, but because one sample can't estimate it). Treat points
with `n_runs == 1` as provisional rather than averaged-out.

In [ ]:
df_train_summary = (
    df_exp.groupby(["circuit_num", "n_qubits", "layers", "anzats_reps"])
    .agg(
        n_runs=("id", "count"),
        n_params=("n_params", "first"),
        mean_final_cost=("final_cost_avg10", "mean"),
        std_final_cost=("final_cost_avg10", "std"),
        sources=("source", lambda s: ",".join(sorted(set(s)))),
    )
    .reset_index()
)
df_train_summary["sem_final_cost"] = (
    df_train_summary["std_final_cost"] / np.sqrt(df_train_summary["n_runs"])
).fillna(0)
df_train_summary.sort_values(["circuit_num", "anzats_reps"]).head(15)

### Final cost vs. number of parameters

One point per architecture (pooled over problem instances, see above), not
one per run -- averaging first avoids pseudo-replication (several rows that
are really the same measurement repeated). Color is `circuit_num`; hover for
the exact config and how many instances it's pooled from.

In [ ]:
plottable = df_train_summary.dropna(subset=["n_params"])
circuit_nums = sorted(plottable["circuit_num"].unique())
palette = {c: plt.cm.tab20(i % 20) for i, c in enumerate(circuit_nums)}

plt.close("all")
fig, ax = plt.subplots(figsize=(9, 6))
handles = []
show_error_bars = False
for c in circuit_nums:
    pts = plottable[plottable["circuit_num"] == c]
    if len(pts):
        if show_error_bars:
            h = ax.errorbar(pts["n_params"], pts["mean_final_cost"], yerr=pts["sem_final_cost"],
                            fmt="o", color=palette[c], alpha=0.8, label=f"circuit {c}", capsize=3)
            handles.append((h.lines[0], pts))
        else:
            h = ax.scatter(pts["n_params"], pts["mean_final_cost"], color=palette[c], alpha=0.8, label=f"circuit {c}")
            handles.append((h, pts))
ax.set_yscale("log")
ax.set_xlabel("Number of parameters",fontsize=14)
ax.set_ylabel("Mean final cost",fontsize=14)

ax.set_title("Final cost vs. number of parameters",fontsize=20)

highlight_promising = False
if highlight_promising:
    # The_good_ones is keyed by circuit_num (architecture-level, from Part A's
    # regression) -- every point of a promising architecture gets ringed,
    # not just a specific (layers, reps) combo of it.
    promising_ids = set(The_good_ones["circuit_num"])
    promising_pts = plottable[plottable["circuit_num"].isin(promising_ids)]
    if len(promising_pts):
        ring_handle = ax.scatter(
            promising_pts["n_params"], promising_pts["mean_final_cost"],
            facecolors="none", edgecolors="black", linewidths=1.8, s=180, zorder=5,
            label=f"promising (k-criteria, {len(promising_ids)} circuits)",
        )
        # explicit handles= here -- a bare ax.legend() would also pick up
        # every per-circuit `label="circuit {c}"` set above (never shown
        # before, since legend() was never called), flooding the legend
        # with 20+ entries nobody asked for.
        ax.legend(handles=[ring_handle], loc="best", fontsize=9)
    else:
        print("no promising-circuit points fall in this plot's (n_params < 1e4) range")


fig.tight_layout()

cursor = mplcursors.cursor([h for h, _ in handles], hover=True)

@cursor.connect("add")
def _(sel):
    pts = next(p for h, p in handles if h is sel.artist)
    row = pts.iloc[sel.index]
    sel.annotation.set_text(
        f"circuit {row.circuit_num}  n_qubits={row.n_qubits}  layers={row.layers}  "
        f"reps={row.anzats_reps}  n_runs={row.n_runs} ({row.sources})\n"
        f"mean_final_cost={row.mean_final_cost:.3g}"
    )

plt.show()

### Cost curves for one circuit

Every recorded run for a chosen `circuit_num`, across `(layers, anzats_reps,
source)` -- individual trajectories, not pooled, so you can see the actual
training noise the mean above is summarizing.

In [ ]:
pick_circuit = 21410
max_steps = 200
#cut the curves at max_steps, because some curves are very long and make the plot less readable

plt.close("all")
fig, ax = plt.subplots(figsize=(9, 6))
rows = df_exp[df_exp["circuit_num"] == pick_circuit][df_exp["anzats_reps"] > 5]
handles_cc = []
for _, row in rows.iterrows():
    curve = df_costs[df_costs["experiment_id"] == row["id"]]
    if len(curve):
        costs = curve.iloc[0]["costs"]
        line, = ax.plot(costs[:max_steps], alpha=0.8,
                         label=f"layers={row.layers} reps={row.anzats_reps} ({row.source})")
        handles_cc.append((line, row))
ax.set_yscale("log")
ax.set_xlabel("step")
ax.set_ylabel("cost")
ax.set_title(f"Cost curves, circuit {pick_circuit}")
ax.legend(fontsize=7)
fig.tight_layout()

cursor_cc = mplcursors.cursor([h for h, _ in handles_cc], hover=True)

@cursor_cc.connect("add")
def _(sel):
    row = next(r for h, r in handles_cc if h is sel.artist)
    sel.annotation.set_text(
        f"circuit {row.circuit_num}  layers={row.layers}  reps={row.anzats_reps}  ({row.source})\n"
        f"final_cost={row.final_cost:.3g}"
    )

plt.show()

## Part C -- Training performance vs. frame potential

Joins `df_train_summary` (one pooled point per `(circuit_num, n_qubits,
layers, anzats_reps)`) to `df_fp` on `(circuit_num, n_qubits, reps)` --
frame potential has no `layers` concept (it's just `circuit_set(num,
reps)`, never stacked with data re-uploading), so every `layers` value for
the same `(circuit_num, n_qubits, anzats_reps)` maps to the *same* `delta`
and shows up as a vertical stack of points at that x -- that's expected, not
a bug, and it's also why `delta` alone is a weaker descriptor of the full
*trained model* than `n_params` is: it only describes one repeated block,
not how many times `layers` stacks it.

In [ ]:
fp_summary = (
    df_fp.groupby(["circuit_num", "n_qubits", "reps"])["delta"]
    .mean()
    .reset_index()
    .rename(columns={"reps": "anzats_reps"})
)

df_joined = pd.merge(df_train_summary, fp_summary, on=["circuit_num", "n_qubits", "anzats_reps"], how="inner")
print(f"{len(df_joined)}/{len(df_train_summary)} training configs matched a frame-potential measurement")
df_joined.sort_values(["circuit_num", "anzats_reps"]).head(15)

### Final cost vs. frame potential difference

`highlight_promising` (below) rings every point belonging to a circuit in
`The_good_ones` (the k-criteria table from Part A) in black, so you can see
directly whether the architectures flagged as "fast, trustworthy Δ*F*^(t)
convergence" also land where cost is actually low -- rather than having to
cross-reference the two tables by hand.</cell id="331afa0f">


In [ ]:
#remove point with Frame potential difference from Haar (delta) bigger than 1e2, because they are outliers and make the plot less readable
plottable_j = df_joined[df_joined["delta"] < 1e4]

plottable_j = plottable_j.dropna(subset=["delta", "mean_final_cost"])
circuit_nums_j = sorted(plottable_j["circuit_num"].unique())
palette_j = {c: plt.cm.tab20(i % 20) for i, c in enumerate(circuit_nums_j)}

highlight_promising = True  # ring-highlight circuits flagged by the k-criteria (The_good_ones, Part A) above

plt.close("all")
fig, ax = plt.subplots(figsize=(9, 6))
handles_j = []
scatter = True
for c in circuit_nums_j:
    pts = plottable_j[plottable_j["circuit_num"] == c]
    if len(pts):
        if scatter:
            h = ax.scatter(pts["delta"], pts["mean_final_cost"], color=palette_j[c], alpha=0.8, label=f"circuit {c}")
            handles_j.append((h, pts))
        else:
            h = ax.errorbar(pts["delta"], pts["mean_final_cost"], yerr=pts["sem_final_cost"],
                         fmt="o", color=palette_j[c], alpha=0.8, label=f"circuit {c}", capsize=3)
            handles_j.append((h.lines[0], pts))

if highlight_promising:
    # The_good_ones is keyed by circuit_num (architecture-level, from Part A's
    # regression) -- every point of a promising architecture gets ringed,
    # not just a specific (layers, reps) combo of it.
    promising_ids = set(The_good_ones["circuit_num"])
    promising_pts = plottable_j[plottable_j["circuit_num"].isin(promising_ids)]
    if len(promising_pts):
        ring_handle = ax.scatter(
            promising_pts["delta"], promising_pts["mean_final_cost"],
            facecolors="none", edgecolors="black", linewidths=1.8, s=180, zorder=5,
            label=f"promising (k-criteria, {len(promising_ids)} circuits)",
        )
        # explicit handles= here -- a bare ax.legend() would also pick up
        # every per-circuit `label="circuit {c}"` set above (never shown
        # before, since legend() was never called), flooding the legend
        # with 20+ entries nobody asked for.
        ax.legend(handles=[ring_handle], loc="best", fontsize=9)
    else:
        print("no promising-circuit points fall in this plot's (delta < 1e4) range")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("$\\Delta F^{(t)}$ (frame potential difference from Haar)",fontsize=14)
ax.set_ylabel("mean final cost",fontsize=14)
ax.set_title("Final cost vs. frame potential difference",fontsize=20)
fig.tight_layout()

cursor_j = mplcursors.cursor([h for h, _ in handles_j], hover=True)

@cursor_j.connect("add")
def _(sel):
    pts = next(p for h, p in handles_j if h is sel.artist)
    row = pts.iloc[sel.index]
    sel.annotation.set_text(
        f"circuit {row.circuit_num}  n_qubits={row.n_qubits}  layers={row.layers}  reps={row.anzats_reps}\n"
        f"delta={row.delta:.3g}  mean_final_cost={row.mean_final_cost:.3g}  n_params={row.n_params}"
    )

plt.show()

### Filling in the circuits missing from the join

`df_joined` above only has a point where a frame-potential measurement
exists at that exact `(circuit_num, n_qubits, anzats_reps)` -- 104 trained
configs have no such measurement and were silently dropped from the plot
above. For a circuit whose `circuit_num` regressed in `k_df` (>=2 distinct
measured `n_params`), that circuit's own fit
(`log(delta) ~= intercept - k * n_params`) gives a stand-in delta at the
missing reps instead.

**Units bug caught while building this**: `df_train_summary`'s `n_params`
already has the `layers` factor baked in
(`n_params = layers * n_trainable(...)`, see `tracking.py`),
while `k_df`'s fit was built against frame potential's own `n_params`,
which has no `layers` concept at all -- just `n_trainable` for one
`circuit_set(num, reps)` call. Dividing by `layers` before evaluating the
fit is what makes the two comparable; skipping that step silently
extrapolated 3x further than intended.

**Extrapolation still matters even after that fix.** None of the 104
missing configs fall *inside* the `n_params` range their circuit's fit was
built on -- frame-potential sweeps stopped at lower reps than training went
to, for every circuit here. The log-linear model has no floor, so pushed
far enough it predicts deltas nothing in this dataset ever measured (the
real minimum is ~5e-5; unconstrained extrapolation reaches 1e-56).
`EXTRAPOLATION_CAP` bounds how far past the fitted range an estimate is
still shown -- beyond that multiple, the point is dropped and counted
rather than plotted as if it were real. Kept points are marked separately
from measured ones (hollow square vs. filled circle) so the two are never
mistaken for each other.

In [ ]:
EXTRAPOLATION_CAP = 2  # how many multiples of the fitted n_params range an estimate may extrapolate before being dropped

missing = pd.merge(
    df_train_summary, fp_summary, on=["circuit_num", "n_qubits", "anzats_reps"],
    how="left", indicator=True,
)
missing = missing[missing["_merge"] == "left_only"].drop(columns=["_merge", "delta"])

k_lookup = k_df.set_index("circuit_num")[["k", "intercept", "x_min", "x_max"]]
missing = missing.join(k_lookup, on="circuit_num")

estimated = missing.dropna(subset=["k", "intercept"]).copy()
n_no_k = len(missing) - len(estimated)

# frame potential's n_params has no `layers` factor -- divide it out before
# evaluating a fit that was built against frame potential's own n_params
estimated["n_params_per_layer"] = estimated["n_params"] / estimated["layers"]
estimated["delta"] = np.exp(estimated["intercept"] - estimated["k"] * estimated["n_params_per_layer"])
estimated["in_fit_range"] = estimated["n_params_per_layer"].between(estimated["x_min"], estimated["x_max"])

within_cap = (
    (estimated["n_params_per_layer"] >= estimated["x_min"] / EXTRAPOLATION_CAP)
    & (estimated["n_params_per_layer"] <= estimated["x_max"] * EXTRAPOLATION_CAP)
)
dropped = estimated[~within_cap]
estimated = estimated[within_cap].copy()

print(f"{len(missing)} training configs have no frame-potential match in the join")
print(f"  {n_no_k} skipped -- that circuit_num never regressed in k_df "
      "(fewer than 2 distinct n_params worth of real delta measurements)")
print(f"  {len(dropped)} skipped -- more than {EXTRAPOLATION_CAP}x past the fitted n_params "
      "range (too far from any real measurement to trust)")
print(f"  {len(estimated)} kept, of which {int(estimated['in_fit_range'].sum())} are genuine "
      f"interpolation and {int((~estimated['in_fit_range']).sum())} are extrapolation within "
      "the cap")
if len(dropped):
    print("\ndropped (circuit_num: n_params_per_layer vs. fitted range):")
    for _, r in dropped.sort_values("circuit_num").iterrows():
        print(f"  circuit {r.circuit_num}: {r.n_params_per_layer:.0f}  "
              f"(fitted on [{r.x_min:.0f}, {r.x_max:.0f}])")

estimated["estimated"] = True
actual = df_joined.copy()
actual["estimated"] = False
actual["in_fit_range"] = True  # not applicable -- measured directly, no fit involved

combined = pd.concat(
    [actual, estimated.drop(columns=["k", "intercept", "x_min", "x_max", "n_params_per_layer"])],
    ignore_index=True, sort=False,
)
combined = combined.dropna(subset=["delta", "mean_final_cost"])

In [ ]:
plottable_ext = combined[combined["delta"] < 1e4].copy()
circuit_nums_ext = sorted(plottable_ext["circuit_num"].unique())
palette_ext = {c: plt.cm.tab20(i % 20) for i, c in enumerate(circuit_nums_ext)}

plt.close("all")
fig, ax = plt.subplots(figsize=(9, 6))
handles_ext = []
for c in circuit_nums_ext:
    pts_c = plottable_ext[plottable_ext["circuit_num"] == c]

    pts_m = pts_c[~pts_c["estimated"]]
    if len(pts_m):
        h = ax.scatter(pts_m["delta"], pts_m["mean_final_cost"], color=palette_ext[c],
                        marker="o", alpha=0.8, s=40)
        handles_ext.append((h, pts_m))

    pts_i = pts_c[pts_c["estimated"] & pts_c["in_fit_range"]]
    if len(pts_i):
        h = ax.scatter(pts_i["delta"], pts_i["mean_final_cost"], facecolors="none",
                        edgecolors=palette_ext[c], marker="^", linewidths=1.4, s=70, alpha=0.9)
        handles_ext.append((h, pts_i))

    pts_e = pts_c[pts_c["estimated"] & ~pts_c["in_fit_range"]]
    if len(pts_e):
        h = ax.scatter(pts_e["delta"], pts_e["mean_final_cost"], facecolors="none",
                        edgecolors=palette_ext[c], marker="s", linewidths=1.3, s=60, alpha=0.6)
        handles_ext.append((h, pts_e))

# proxy handles only -- a per-circuit legend here would be the same 20+-entry
# flood noted in the plot above; this legend exists only to explain the markers.
proxies = [
    ax.scatter([], [], marker="o", color="gray", label="measured (in the join)"),
    ax.scatter([], [], marker="^", facecolors="none", edgecolors="gray", label="estimated, interpolated"),
    ax.scatter([], [], marker="s", facecolors="none", edgecolors="gray", alpha=0.6,
               label=f"estimated, extrapolated (<{EXTRAPOLATION_CAP}x)"),
]
ax.legend(handles=proxies, loc="best", fontsize=9)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("$\\Delta F^{(t)}$ (measured, or k-regression estimate)", fontsize=14)
ax.set_ylabel("mean final cost", fontsize=14)
ax.set_title("Final cost vs. frame potential -- measured + k-estimated", fontsize=18)
fig.tight_layout()

cursor_ext = mplcursors.cursor([h for h, _ in handles_ext], hover=True)

@cursor_ext.connect("add")
def _(sel):
    pts = next(p for h, p in handles_ext if h is sel.artist)
    row = pts.iloc[sel.index]
    tag = "estimated" if row.estimated else "measured"
    sel.annotation.set_text(
        f"circuit {row.circuit_num}  n_qubits={row.n_qubits}  layers={row.layers}  reps={row.anzats_reps}  [{tag}]\n"
        f"delta={row.delta:.3g}  mean_final_cost={row.mean_final_cost:.3g}  n_params={row.n_params}"
    )

plt.show()

## Part D -- Frame potential vs. parameter count as performance indicators

**Method.** Both `n_params` and `delta` are candidate single-number
predictors of `mean_final_cost`, and the relationship between either of them
and cost is expected to be monotonic rather than linear (cost spans orders
of magnitude; so do both predictors) -- **Spearman rank correlation**
(`scipy.stats.spearmanr`) is the right tool: it only assumes monotonicity,
ignores the actual scale/shape of the relationship, and is robust to the
handful of outlier runs a sweep like this tends to have. Pearson correlation
on raw values would conflate "monotonic but curved" with "no relationship".

Comparing the two `rho` values directly is a reasonable first cut, but can
be misleading: `n_params` and `delta` are themselves correlated (bigger
circuits tend to also be closer to a 2-design), so a high `rho(cost,
delta)` could just be riding on `n_params`'s coattails. The sharper
question -- does `delta` carry information about performance *beyond* what
parameter count already gives you -- is a **partial Spearman correlation**:
rank-transform all three variables, linearly regress `n_params`'s ranks out
of both `cost`'s ranks and `delta`'s ranks, then correlate the residuals
(this is the standard rank-residual way to compute a partial Spearman
correlation; computing it the other way around, controlling for `delta`
instead, answers the complementary question of whether raw parameter count
still matters once `delta` is accounted for).

One row per architecture (`df_joined`, already pooled over problem
instances in Part B) is the right unit here -- correlating against raw
per-run costs would pseudo-replicate every point `n_runs` times without
adding real information about the cost-vs-delta relationship.

**Caveat worth keeping in mind**: with ~20 joined architectures (and most of
them at a single fixed `n_qubits`/`layers` so far), these correlations are
low-powered -- read `rho` as a rough signal, not a settled conclusion, and
revisit once `pull_remote_results.sh` has brought in more runs.

In [ ]:
df_joined

In [ ]:
def partial_spearman(x, y, z):
    """Spearman partial correlation between x and y, controlling for z:
    rank all three, linearly regress z's ranks out of x's and y's ranks,
    then Pearson-correlate the residuals (equivalent to a partial Spearman)."""
    rx, ry, rz = rankdata(x), rankdata(y), rankdata(z)

    def residualize(a, b):
        slope, intercept = np.polyfit(b, a, 1)
        return a - (slope * b + intercept)

    return pearsonr(residualize(rx, rz), residualize(ry, rz))


corr_data = df_joined.dropna(subset=["mean_final_cost", "n_params", "delta"])
cost, n_params, delta = corr_data["mean_final_cost"], corr_data["n_params"], corr_data["delta"]
n = len(corr_data)

rho_params, p_params = spearmanr(cost, n_params)
rho_delta, p_delta = spearmanr(cost, delta)
rho_delta_given_params, p_dgp = partial_spearman(cost, delta, n_params)
rho_params_given_delta, p_pgd = partial_spearman(cost, n_params, delta)

print(f"n = {n} architectures")
print(f"rho(cost, n_params)                 = {rho_params:+.3f}  (p={p_params:.3f})")
print(f"rho(cost, delta)                    = {rho_delta:+.3f}  (p={p_delta:.3f})")
print(f"rho(cost, delta   | n_params)       = {rho_delta_given_params:+.3f}  (p={p_dgp:.3f})")
print(f"rho(cost, n_params | delta)         = {rho_params_given_delta:+.3f}  (p={p_pgd:.3f})")

## Part E -- Correlation confound: circuit class (RY + real entangler = cosine-only)

**Reviewer note.** Data encoding is `RX(x)`. If every other gate in the
circuit is real -- RY rotations (a Y-rotation is a real orthogonal matrix)
and a real entangler (none, CNOT, CZ, or CRY; **not** CRX/CRZ, and **not**
RX/RZ/`Rot` rotations, all of which are complex -- see `generic_ansatz.py`'s
module docstring) -- then complex-conjugating the whole circuit sends
`RX(x) -> RX(-x)` and leaves everything else fixed, so
`|psi(x)>* = |psi(-x)>`. With the real (`PauliZ`) observable this codebase
measures (`functions.py`'s `build_model`), that forces the trained output to
be an *even* function of `x`: these circuits can only fit the cosine part of
a complex-coefficient Fourier target, no matter how well they train -- a
hard expressibility ceiling, not noise.

`is_real_valued_circuit` below flags exactly this class from `circuit_num`
via `generic_ansatz.decode`. It's 38% of `df_joined`'s points (17 of 91
circuits), sits almost entirely at `delta >= 1`, and forms the horizontal
band at `cost ~ 1e-2` visible once the cost-vs-delta plot is colored by
class -- splitting `rho(cost, delta)` the same way (Table 2) shows the
correlation holds within the unconstrained class (+0.67) and collapses to
noise within the cosine-only one (-0.06): it was never one trend, it's two
populations with a confound.

In [ ]:
def is_real_valued_circuit(code: int) -> bool:
    """RY rotation + a real entangler (none, CNOT, CZ, or CRY) -- see
    generic_ansatz.py's module docstring for which gates are real. Any
    other rotation_arity (RX/RZ or full Rot) or entangler (CRX/CRZ) is
    complex, so the parity argument above doesn't apply."""
    spec = generic_ansatz.decode(code)
    if spec.rotation_arity != 1:
        return False
    if spec.gate_category == 0:
        return True
    if spec.gate_category == 1:
        return True  # CNOT or CZ -- both real
    return spec.gate_type == 1  # parametrized: only CRY (gate_type 1) is real


df_joined["is_real"] = df_joined["circuit_num"].apply(is_real_valued_circuit)

n_real_pts = int(df_joined["is_real"].sum())
n_real_circuits = df_joined.loc[df_joined["is_real"], "circuit_num"].nunique()
n_complex_circuits = df_joined.loc[~df_joined["is_real"], "circuit_num"].nunique()
print(f"{n_real_pts}/{len(df_joined)} points ({n_real_pts / len(df_joined):.0%}) are cosine-only "
      f"(RY + real entangler): {n_real_circuits} circuits, vs {n_complex_circuits} unconstrained circuits")

In [ ]:
plottable_cls = df_joined.dropna(subset=["delta", "mean_final_cost"])
plottable_cls = plottable_cls[plottable_cls["delta"] < 1e4]

plt.close("all")
fig, ax = plt.subplots(figsize=(9, 6))
handles_cls = []
for is_real, color, label in [
    (False, "tab:blue", "unconstrained"),
    (True, "tab:red", "cosine-only (RY + real entangler)"),
]:
    pts = plottable_cls[plottable_cls["is_real"] == is_real]
    if len(pts):
        h = ax.scatter(pts["delta"], pts["mean_final_cost"], color=color, alpha=0.6, s=35, label=label)
        handles_cls.append((h, pts))

ax.axvline(1.0, color="black", linestyle="--", linewidth=1, alpha=0.6)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("$\\Delta F^{(t)}$ (frame potential difference from Haar)", fontsize=14)
ax.set_ylabel("mean final cost", fontsize=14)
ax.set_title("Final cost vs. frame potential -- split by circuit class", fontsize=16)
ax.legend(loc="best", fontsize=9)
fig.tight_layout()

cursor_cls = mplcursors.cursor([h for h, _ in handles_cls], hover=True)

@cursor_cls.connect("add")
def _(sel):
    pts = next(p for h, p in handles_cls if h is sel.artist)
    row = pts.iloc[sel.index]
    cls = "cosine-only" if row.is_real else "unconstrained"
    sel.annotation.set_text(
        f"circuit {row.circuit_num}  [{cls}]  n_qubits={row.n_qubits}  layers={row.layers}  reps={row.anzats_reps}\n"
        f"delta={row.delta:.3g}  mean_final_cost={row.mean_final_cost:.3g}"
    )

plt.show()

In [ ]:
plottable_cls_p = df_joined.dropna(subset=["n_params", "mean_final_cost"])

plt.close("all")
fig, ax = plt.subplots(figsize=(9, 6))
handles_cls_p = []
for is_real, color, label in [
    (False, "tab:blue", "unconstrained"),
    (True, "tab:red", "cosine-only (RY + real entangler)"),
]:
    pts = plottable_cls_p[plottable_cls_p["is_real"] == is_real]
    if len(pts):
        h = ax.scatter(pts["n_params"], pts["mean_final_cost"], color=color, alpha=0.6, s=35, label=label)
        handles_cls_p.append((h, pts))

ax.set_yscale("log")
ax.set_xlabel("number of parameters", fontsize=14)
ax.set_ylabel("mean final cost", fontsize=14)
ax.set_title("Final cost vs. number of parameters -- split by circuit class", fontsize=16)
ax.legend(loc="best", fontsize=9)
fig.tight_layout()

cursor_cls_p = mplcursors.cursor([h for h, _ in handles_cls_p], hover=True)

@cursor_cls_p.connect("add")
def _(sel):
    pts = next(p for h, p in handles_cls_p if h is sel.artist)
    row = pts.iloc[sel.index]
    cls = "cosine-only" if row.is_real else "unconstrained"
    sel.annotation.set_text(
        f"circuit {row.circuit_num}  [{cls}]  n_qubits={row.n_qubits}  layers={row.layers}  reps={row.anzats_reps}\n"
        f"n_params={row.n_params}  mean_final_cost={row.mean_final_cost:.3g}"
    )

plt.show()

In [ ]:
rows_cls = []
for label, mask in [
    ("all circuits", pd.Series(True, index=df_joined.index)),
    ("cosine-only (RY + real entangler)", df_joined["is_real"]),
    ("unconstrained", ~df_joined["is_real"]),
]:
    d = df_joined[mask].dropna(subset=["mean_final_cost", "n_params", "delta"])
    rho_delta, p_delta = spearmanr(d["mean_final_cost"], d["delta"])
    rho_params, p_params = spearmanr(d["mean_final_cost"], d["n_params"])
    rho_delta_given_params, p_dgp = partial_spearman(d["mean_final_cost"], d["delta"], d["n_params"])
    rows_cls.append({
        "class": label, "n_points": len(d), "pct_of_total": len(d) / len(df_joined) * 100,
        "n_circuits": d["circuit_num"].nunique(),
        "rho_cost_delta": rho_delta, "p_value_delta": p_delta,
        "rho_cost_nparams": rho_params, "p_value_nparams": p_params,
        "rho_cost_delta_given_params": rho_delta_given_params, "p_value_delta_given_params": p_dgp,
    })

class_table = pd.DataFrame(rows_cls)
class_table.style.format({
    "pct_of_total": "{:.1f}%",
    "rho_cost_delta": "{:+.3f}", "p_value_delta": "{:.2g}",
    "rho_cost_nparams": "{:+.3f}", "p_value_nparams": "{:.2g}",
    "rho_cost_delta_given_params": "{:+.3f}", "p_value_delta_given_params": "{:.2g}",
}).hide(axis="index").set_caption("Table 2 -- correlation within circuit class (delta and n_params)")